In [1]:
from torch_geometric.datasets import Amazon
import numpy as np
import torch

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
device

/Users/zhangyue/Workspace/github/learning-toy-examples/.venv/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


device(type='cpu')

In [2]:
# classifying the products into different categories
dataset = Amazon(root='data/amazon', name='Computers')

print(dataset)
print(dataset[0])
print(dataset[0].x.shape)
print(dataset[0].y.shape)
print(dataset[0].edge_index.shape)

AmazonComputers()
Data(x=[13752, 767], edge_index=[2, 491722], y=[13752])
torch.Size([13752, 767])
torch.Size([13752])
torch.Size([2, 491722])


In [3]:
np.unique(dataset[0].y)

array([0, 1, 2, 3, 4, 5, 6, 7, 8, 9])

# Model

In [4]:
import torch
import torch.nn.functional as F
import torch.nn as nn
from torch_geometric.nn import GCNConv
from sklearn.model_selection import StratifiedKFold

class GCN(torch.nn.Module):
    def __init__(self, in_channels, hidden_channels, out_channels, dropout=0.2):
        super().__init__()
        self.conv1 = GCNConv(in_channels, hidden_channels)
        self.conv2 = GCNConv(hidden_channels, out_channels)
        self.dropout = nn.Dropout(dropout)

    def forward(self, x, edge_index):
        x = self.conv1(x, edge_index)
        x = F.relu(x)
        x = self.dropout(x)
        return self.conv2(x, edge_index)

In [9]:
def run_epoch(model, optimizer, data, train_mask, val_mask):
    model.train()
    optimizer.zero_grad()
    out = model(data.x, data.edge_index)
    train_loss = F.cross_entropy(out[train_mask], data.y[train_mask])
    train_loss.backward()
    optimizer.step()

    model.eval()
    with torch.no_grad():
        out = model(data.x, data.edge_index)
        val_loss = F.cross_entropy(out[val_mask], data.y[val_mask])
    return train_loss.item(), val_loss.item()


@torch.no_grad()
def masked_accuracy(model, data, mask):
    model.eval()
    out = model(data.x, data.edge_index)
    pred = out[mask].argmax(dim=-1)
    return (pred == data.y[mask]).float().mean().item()


data = dataset[0].to(device)
num_nodes = data.num_nodes
num_classes = int(data.y.max().item()) + 1
in_channels = data.num_node_features
hidden_channels = 64
max_epochs = 10
patience = 20  # early stop after this many epochs without val improvement
min_delta = 1e-4
lr = 0.005
weight_decay = 5e-4  # L2 penalty in Adam; works with dropout against overfitting
# Reduce LR when val loss stalls; stop training only after more patience (see loop below)
lr_scheduler_patience = 15
lr_factor = 0.5
min_lr = 1e-5
n_folds = 5

skf = StratifiedKFold(n_splits=n_folds, shuffle=True, random_state=42)
fold_history = {}

for fold_idx, (train_idx, val_idx) in enumerate(skf.split(np.arange(num_nodes), data.y.cpu().numpy())):
    train_mask = torch.zeros(num_nodes, dtype=torch.bool, device=device)
    val_mask = torch.zeros(num_nodes, dtype=torch.bool, device=device)
    train_mask[train_idx] = True
    val_mask[val_idx] = True

    model = GCN(in_channels, hidden_channels, num_classes).to(device)
    optimizer = torch.optim.Adam(
        model.parameters(), lr=lr, weight_decay=weight_decay
    )
    scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
        optimizer,
        mode="min",
        factor=lr_factor,
        patience=lr_scheduler_patience,
        min_lr=min_lr,
        threshold=min_delta,
    )

    train_losses, val_losses, lr_history = [], [], []
    best_val = float("inf")
    epochs_without_improvement = 0
    best_state = None

    for epoch in range(max_epochs):
        train_loss, val_loss = run_epoch(model, optimizer, data, train_mask, val_mask)
        train_losses.append(train_loss)
        val_losses.append(val_loss)
        scheduler.step(val_loss)
        lr_history.append(optimizer.param_groups[0]["lr"])

        if val_loss < best_val - min_delta:
            best_val = val_loss
            epochs_without_improvement = 0
            best_state = {k: v.detach().cpu().clone() for k, v in model.state_dict().items()}
        else:
            epochs_without_improvement += 1
            if epochs_without_improvement >= patience:
                break

    if best_state is not None:
        model.load_state_dict({k: v.to(device) for k, v in best_state.items()})

    train_acc = masked_accuracy(model, data, train_mask)
    val_acc = masked_accuracy(model, data, val_mask)

    fold_history[f"fold_{fold_idx}"] = {
        "train_loss": train_losses,
        "val_loss": val_losses,
        "lr_history": lr_history,
        "best_val_loss": best_val,
        "epochs_trained": len(train_losses),
        "final_lr": lr_history[-1],
        "train_acc": train_acc,
        "val_acc": val_acc,
    }
    print(
        f"fold {fold_idx}: stopped at epoch {len(train_losses)}/{max_epochs}, "
        f"best val_loss={best_val:.4f}, "
        f"train_acc={train_acc:.4f}, val_acc={val_acc:.4f}, "
        f"final_lr={lr_history[-1]:.2e}"
    )

train_accs = [fold_history[f"fold_{i}"]["train_acc"] for i in range(n_folds)]
val_accs = [fold_history[f"fold_{i}"]["val_acc"] for i in range(n_folds)]
print(
    f"\nCV summary: train_acc={np.mean(train_accs):.4f} ± {np.std(train_accs):.4f}, "
    f"val_acc={np.mean(val_accs):.4f} ± {np.std(val_accs):.4f}"
)

# things to check
# - overfitting: train_loss < val_loss? if overfitting, consider dropouts + weight_decay(L2 regularization, encourages smaller weights)
# - learning rate: too high or too low? LR scheduler for plateau
# - number of epochs: too many or too few?


fold 0: stopped at epoch 10/10, best val_loss=1.6999, train_acc=0.5247, val_acc=0.5202, final_lr=5.00e-03
fold 1: stopped at epoch 10/10, best val_loss=1.6686, train_acc=0.5111, val_acc=0.5056, final_lr=5.00e-03
fold 2: stopped at epoch 10/10, best val_loss=1.6904, train_acc=0.6659, val_acc=0.6575, final_lr=5.00e-03
fold 3: stopped at epoch 10/10, best val_loss=1.7176, train_acc=0.6333, val_acc=0.6393, final_lr=5.00e-03
fold 4: stopped at epoch 10/10, best val_loss=1.7724, train_acc=0.5177, val_acc=0.5222, final_lr=5.00e-03

CV summary: train_acc=0.5706 ± 0.0655, val_acc=0.5689 ± 0.0654


## Mini-batch training (large graphs)

For **node classification**, use `NeighborLoader` (k-hop subgraphs per batch).

1. Run the **setup cell below** once per venv (installs `torch==2.5.1` + `torch-sparse` wheels for macOS CPU).
2. Run the **full-batch** cell above (`GCN`, `skf`, hyperparameters).
3. Run the **minibatch training** cell.

`LinkNeighborLoader` is for link prediction, not this node-label task.

In [5]:
# One-time setup for NeighborLoader (macOS / CPU). Re-run after recreating .venv.
import importlib.util
import subprocess
import sys

PYG_CPU_INDEX = "https://data.pyg.org/whl/torch-2.5.1+cpu.html"
TORCH_VERSION = "2.5.1"


def neighbor_sampling_ready():
    return importlib.util.find_spec("torch_sparse") is not None


if not neighbor_sampling_ready():
    print("Installing torch 2.5.1 + torch-scatter + torch-sparse...")
    subprocess.check_call(
        [sys.executable, "-m", "pip", "install", f"torch=={TORCH_VERSION}"],
    )
    subprocess.check_call(
        [
            sys.executable,
            "-m",
            "pip",
            "install",
            "torch-scatter",
            "torch-sparse",
            "-f",
            PYG_CPU_INDEX,
        ],
    )
    print("Done. Restart the notebook kernel, then re-run this cell.")
else:
    import torch
    import torch_sparse  # noqa: F401 — required by NeighborLoader
    from torch_geometric.loader import NeighborLoader

    print(f"torch {torch.__version__}, torch_sparse {torch_sparse.__version__}")
    print("NeighborLoader is ready.")

torch 2.5.1, torch_sparse 0.6.18
NeighborLoader is ready.


In [11]:
from torch_geometric.loader import NeighborLoader

try:
    import torch_sparse  # noqa: F401 — NeighborLoader backend
except ImportError as e:
    raise ImportError(
        "Run the setup cell above first (installs torch 2.5.1 + torch-sparse)."
    ) from e

# True = short run to learn the flow; False = same epoch budget as full-batch cell
MINIBATCH_QUICK_DEMO = True
epochs_to_run = 15 if MINIBATCH_QUICK_DEMO else max_epochs
patience_mb = 5 if MINIBATCH_QUICK_DEMO else patience

# 2-layer GCN → sample neighbors for 2 hops (tune for memory vs receptive field)
num_neighbors = [15, 10]
batch_size = 1024  # seed nodes per step; lower if OOM

def make_loader(data, input_nodes, shuffle):
    return NeighborLoader(
        data,
        num_neighbors=num_neighbors,
        batch_size=batch_size,
        input_nodes=input_nodes,  # boolean index of seed nodes to start with
        shuffle=shuffle,
    )


def train_epoch_minibatch(model, optimizer, loader):
    model.train()
    total_loss, total_count = 0.0, 0
    for batch in loader:
        batch = batch.to(device)
        optimizer.zero_grad()
        out = model(batch.x, batch.edge_index)[: batch.batch_size]
        y = batch.y[: batch.batch_size]
        loss = F.cross_entropy(out, y)
        loss.backward()
        optimizer.step()
        n = batch.batch_size
        total_loss += loss.item() * n
        total_count += n
    return total_loss / total_count


@torch.no_grad()
def eval_epoch_minibatch(model, loader):
    model.eval()
    total_loss, total_count = 0.0, 0
    for batch in loader:
        batch = batch.to(device)
        out = model(batch.x, batch.edge_index)[: batch.batch_size]
        y = batch.y[: batch.batch_size]
        loss = F.cross_entropy(out, y)
        n = batch.batch_size
        total_loss += loss.item() * n
        total_count += n
    return total_loss / total_count


@torch.no_grad()
def batched_accuracy(model, loader):
    model.eval()
    correct, total = 0, 0
    for batch in loader:
        batch = batch.to(device)
        out = model(batch.x, batch.edge_index)[: batch.batch_size]
        pred = out.argmax(dim=-1)
        y = batch.y[: batch.batch_size]
        correct += (pred == y).sum().item()
        total += batch.batch_size
    return correct / total


# start with seed node of 1024, and during training the batch includes nodes span from seed node with 2-hop neighbors, each with up to 15 and 10 neighbors respectively.
data_cpu = dataset[0]
num_nodes = data_cpu.num_nodes
num_classes = int(data_cpu.y.max().item()) + 1
in_channels = data_cpu.num_node_features

fold_history_minibatch = {}

for fold_idx, (train_idx, val_idx) in enumerate(
    skf.split(np.arange(num_nodes), data_cpu.y.numpy())
):
    train_mask = torch.zeros(num_nodes, dtype=torch.bool)
    val_mask = torch.zeros(num_nodes, dtype=torch.bool)
    train_mask[train_idx] = True
    val_mask[val_idx] = True

    train_loader = make_loader(data_cpu, train_mask, shuffle=True)
    val_loader = make_loader(data_cpu, val_mask, shuffle=False)

    model = GCN(in_channels, hidden_channels, num_classes).to(device)
    optimizer = torch.optim.Adam(
        model.parameters(), lr=lr, weight_decay=weight_decay
    )
    scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
        optimizer,
        mode="min",
        factor=lr_factor,
        patience=lr_scheduler_patience,
        min_lr=min_lr,
        threshold=min_delta,
    )

    train_losses, val_losses, lr_history = [], [], []
    best_val = float("inf")
    epochs_without_improvement = 0
    best_state = None

    for epoch in range(epochs_to_run):
        train_loss = train_epoch_minibatch(model, optimizer, train_loader)
        val_loss = eval_epoch_minibatch(model, val_loader)
        train_losses.append(train_loss)
        val_losses.append(val_loss)
        scheduler.step(val_loss)
        lr_history.append(optimizer.param_groups[0]["lr"])

        if val_loss < best_val - min_delta:
            best_val = val_loss
            epochs_without_improvement = 0
            best_state = {k: v.detach().cpu().clone() for k, v in model.state_dict().items()}
        else:
            epochs_without_improvement += 1
            if epochs_without_improvement >= patience_mb:
                break

    if best_state is not None:
        model.load_state_dict({k: v.to(device) for k, v in best_state.items()})

    train_acc = batched_accuracy(model, train_loader)
    val_acc = batched_accuracy(model, val_loader)

    fold_history_minibatch[f"fold_{fold_idx}"] = {
        "train_loss": train_losses,
        "val_loss": val_losses,
        "best_val_loss": best_val,
        "epochs_trained": len(train_losses),
        "train_acc": train_acc,
        "val_acc": val_acc,
        "final_lr": lr_history[-1],
    }
    print(
        f"[minibatch] fold {fold_idx}: epoch {len(train_losses)}/{epochs_to_run}, "
        f"best val_loss={best_val:.4f}, train_acc={train_acc:.4f}, val_acc={val_acc:.4f}"
    )

mb_train = [fold_history_minibatch[f"fold_{i}"]["train_acc"] for i in range(n_folds)]
mb_val = [fold_history_minibatch[f"fold_{i}"]["val_acc"] for i in range(n_folds)]
print(
    f"\n[minibatch] CV: train_acc={np.mean(mb_train):.4f} ± {np.std(mb_train):.4f}, "
    f"val_acc={np.mean(mb_val):.4f} ± {np.std(mb_val):.4f}"
)

[minibatch] fold 0: epoch 15/15, best val_loss=0.4329, train_acc=0.8656, val_acc=0.8604
[minibatch] fold 1: epoch 15/15, best val_loss=0.4644, train_acc=0.8636, val_acc=0.8590
[minibatch] fold 2: epoch 15/15, best val_loss=0.4758, train_acc=0.8595, val_acc=0.8611
[minibatch] fold 3: epoch 15/15, best val_loss=0.4584, train_acc=0.8585, val_acc=0.8567
[minibatch] fold 4: epoch 15/15, best val_loss=0.5033, train_acc=0.8408, val_acc=0.8422

[minibatch] CV: train_acc=0.8576 ± 0.0088, val_acc=0.8559 ± 0.0070
